Resumen del notebook: Se crean las categorías y modelos como listas. Y los costes, calidades, demandas y capacidades como diccionarios. Se definen las variables, el objetivo, las restricciones y se resuelve el problema de optimización con Gurobi

Restricciones inventadas para la distribución de consultas entre los modelos:
-GPT-4 250/día, GPT-3.5 600/día, Mistral 700/día
-1000 consultas diarias resueltas obligatorias (entre las 3 categorías)
-Mínima precisión de cada categoría
-Variables enteras y NO negativas

In [1]:
from pathlib import Path
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

data_dir = Path("data") / "processed"
parameters = pd.read_csv(data_dir / "model_parameters.csv")

display(parameters)

,category,model,mean_cost,accuracy,sample_count
0,mathematics,gpt-3.5-turbo-1106,0.000105,0.392330,678
1,mathematics,gpt-4-1106-preview,0.001061,0.579646,678
2,mathematics,mistralai/mixtral-8x7b-chat,0.000063,0.221239,678
3,programming,gpt-3.5-turbo-1106,0.000338,0.641176,340
4,programming,gpt-4-1106-preview,0.009426,0.682353,340
5,programming,mistralai/mixtral-8x7b-chat,0.000176,0.535294,340
6,science,gpt-3.5-turbo-1106,0.000091,0.830782,1176
7,science,gpt-4-1106-preview,0.000923,0.960884,1176
8,science,mistralai/mixtral-8x7b-chat,0.000054,0.829932,1176


In [ ]:
#Define groups and parameters
# Sets
categories = parameters["category"].unique().tolist()
models = parameters["model"].unique().tolist()

# Parameters estimated from benchmark data
cost = parameters.set_index(
    ["category", "model"]
)["mean_cost"].to_dict()

accuracy = parameters.set_index(
    ["category", "model"]
)["accuracy"].to_dict()

In [4]:
#1000 queries per day between all the three cathegories
demand = {
    "mathematics": 300,
    "programming": 300,
    "science": 400
}

# Synthetic capacity: queries per day
capacity = {
    "gpt-3.5-turbo-1106": 600,
    "gpt-4-1106-preview": 250,
    "mistralai/mixtral-8x7b-chat": 700
}

# Minimum expected accuracy by category (invented for the experiment)
minimum_accuracy = {
    "mathematics": 0.45,
    "programming": 0.60,
    "science": 0.85
}

assert set(categories) == set(demand) == set(minimum_accuracy)
assert set(models) == set(capacity)
assert len(parameters) == len(categories) * len(models)
assert not parameters.duplicated(["category", "model"]).any()

In [5]:
# Create the optimization model
#This cell traduces directly the formulation from the inform: 
    #9 integer variables (one per each catthegory and model)
optimization_model = gp.Model("query_allocation")

# Decision variables: queries assigned to each model
x = optimization_model.addVars(
    categories,
    models,
    vtype=GRB.INTEGER,
    lb=0,
    name="queries"
)

# Objective: minimize total expected cost
optimization_model.setObjective(
    gp.quicksum(
        cost[category, model] * x[category, model]
        for category in categories
        for model in models
    ),
    GRB.MINIMIZE
)

# Serve all demand
for category in categories:
    optimization_model.addConstr(
        gp.quicksum(
            x[category, model] for model in models
        ) == demand[category],
        name=f"demand_{category}"
    )

# Respect model capacities
for model in models:
    optimization_model.addConstr(
        gp.quicksum(
            x[category, model] for category in categories
        ) <= capacity[model],
        name=f"capacity_{model}"
    )

# Meet minimum expected accuracy in each category
for category in categories:
    optimization_model.addConstr(
        gp.quicksum(
            accuracy[category, model] * x[category, model]
            for model in models
        ) >= minimum_accuracy[category] * demand[category],
        name=f"quality_{category}"
    )

optimization_model.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 9 285H, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 9 rows, 9 columns and 27 nonzeros (Min)
Model fingerprint: 0x91eb3d72
Model has 9 linear objective coefficients
Variable types: 0 continuous, 9 integer (0 binary)
Coefficient statistics:
  Matrix range     [2e-01, 1e+00]
  Objective range  [5e-05, 9e-03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+02, 7e+02]

Presolve removed 1 rows and 0 columns
Presolve time: 0.00s
Presolved: 8 rows, 9 columns, 21 nonzeros
Variable types: 0 continuous, 9 integer (0 binary)
Found heuristic solution: objective 0.3724176

Root relaxation: objective 2.777973e-01, 4 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd 

In [6]:
#It shows and proves the results (in the money units of the benchmark)
if optimization_model.Status == GRB.OPTIMAL:
    allocation = pd.DataFrame(
        {
            model: [
                int(round(x[category, model].X))
                for category in categories
            ]
            for model in models
        },
        index=categories
    )
    allocation.index.name = "category"

    print("Optimal query allocation:")
    display(allocation)

    category_results = []

    for category in categories:
        assigned = allocation.loc[category].sum()

        expected_accuracy = sum(
            accuracy[category, model]
            * allocation.loc[category, model]
            for model in models
        ) / demand[category]

        category_cost = sum(
            cost[category, model]
            * allocation.loc[category, model]
            for model in models
        )

        category_results.append({
            "category": category,
            "demand": demand[category],
            "assigned": assigned,
            "minimum_accuracy": minimum_accuracy[category],
            "expected_accuracy": expected_accuracy,
            "expected_cost": category_cost
        })

    results = pd.DataFrame(category_results).set_index("category")

    capacity_check = pd.DataFrame({
        "assigned": allocation.sum(axis=0),
        "capacity": pd.Series(capacity)
    })

    print("Results by category:")
    display(results)

    print("Capacity check:")
    display(capacity_check)

    # Verify the rounded integer allocation
    assert (results["assigned"] == results["demand"]).all()
    assert (
        results["expected_accuracy"] + 1e-6
        >= results["minimum_accuracy"]
    ).all()
    assert (
        capacity_check["assigned"] <= capacity_check["capacity"]
    ).all()

    print(
        "Total expected cost:",
        round(results["expected_cost"].sum(), 6)
    )

elif optimization_model.Status == GRB.INFEASIBLE:
    print("The scenario is infeasible.")

else:
    print("Solver status:", optimization_model.Status)

Optimal query allocation:


,gpt-3.5-turbo-1106,gpt-4-1106-preview,mistralai/mixtral-8x7b-chat
category,,,
mathematics,207,93,0
programming,184,0,116
science,0,62,338


Results by category:


,demand,assigned,minimum_accuracy,expected_accuracy,expected_cost
category,,,,,
mathematics,300,300,0.45,0.450398,0.120411
programming,300,300,0.60,0.600235,0.082564
science,400,400,0.85,0.850230,0.075547


Capacity check:


,assigned,capacity
gpt-3.5-turbo-1106,391,600
gpt-4-1106-preview,155,250
mistralai/mixtral-8x7b-chat,454,700


Total expected cost: 0.278522
